# RS³Mamba seed42 formal Val run
Official RS³Mamba (IEEE GRSL 2024), adapted from RGB to five lunar channels. Test is locked.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, time
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.5.0', 'rasterio', 'einops', 'monai', 'timm==1.0.26'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mamba-ssm==2.3.2.post1', 'causal-conv1d==1.6.2.post1', '--no-build-isolation'])
REPO = Path('/kaggle/working/lunar-linear')
if not (REPO / '.git').is_dir():
    for attempt in range(1, 4):
        if REPO.exists():
            assert REPO.resolve() == Path('/kaggle/working/lunar-linear'), REPO
            shutil.rmtree(REPO)
        clone = subprocess.run(['git', 'clone', '-b', 'test-new-module', 'https://github.com/song110585-cpu/lunar-linear.git', str(REPO)])
        if clone.returncode == 0:
            break
        if attempt == 3:
            clone.check_returncode()
        print(f'clone失败，第{attempt}次重试...')
        time.sleep(5 * attempt)
else:
    subprocess.check_call(['git', 'pull', '--ff-only', 'origin', 'test-new-module'], cwd=REPO)
PROJECT_DIR = REPO / 'LTL-Net'
print('project commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import hashlib, json, torch
CONFIG_PATH = PROJECT_DIR / 'configs/v6_overlap40_rs3mamba_full_seed42.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
OFFICIAL_REPO = Path('/kaggle/working/SSRS-official')
if not (OFFICIAL_REPO / '.git').is_dir():
    subprocess.check_call(['git', 'clone', config['official_repository'], str(OFFICIAL_REPO)])
subprocess.check_call(['git', 'checkout', '--detach', config['official_commit']], cwd=OFFICIAL_REPO)
official_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=OFFICIAL_REPO, text=True).strip()
assert official_commit == config['official_commit'], (official_commit, config['official_commit'])
RS3_SOURCE = OFFICIAL_REPO / 'RS3Mamba'
VMAMBA_WEIGHT = RS3_SOURCE / 'pretrain' / config['vmamba_pretrained_filename']
assert hashlib.sha256(VMAMBA_WEIGHT.read_bytes()).hexdigest() == config['expected_vmamba_pretrained_sha256']
PRETRAIN_DIR = Path('/kaggle/working/pretrain'); PRETRAIN_DIR.mkdir(exist_ok=True)
RESNET_WEIGHT = PRETRAIN_DIR / config['resnet_pretrained_filename']
if not RESNET_WEIGHT.is_file():
    torch.hub.download_url_to_file(config['resnet_pretrained_url'], str(RESNET_WEIGHT), hash_prefix=config['resnet_hash_prefix'], progress=True)
data_paths = [p for p in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40') if p.is_dir() and (p / 'dataset_protocol.json').is_file()]
assert len(data_paths) == 1, f'应唯一找到数据集，实际: {data_paths}'
DATA_DIR = data_paths[0]
for split, expected in config['expected_tiles'].items():
    images = list((DATA_DIR / split / 'image').glob('*.tif')) + list((DATA_DIR / split / 'image').glob('*.tiff'))
    masks = list((DATA_DIR / split / 'mask').glob('*.tif')) + list((DATA_DIR / split / 'mask').glob('*.tiff'))
    assert len(images) == len(masks) == expected, (split, len(images), len(masks), expected)
for filename, expected_hash in config['expected_metadata_sha256'].items():
    assert hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest() == expected_hash, filename
assert config['automatic_test_evaluation'] is False
print('official commit:', official_commit[:8])
print('data:', DATA_DIR)
print('VMamba weight:', VMAMBA_WEIGHT)
print('ResNet18 weight:', RESNET_WEIGHT)

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
from models.rs3mamba_official_adapter import build_rs3mamba
smoke_model, loading = build_rs3mamba(RS3_SOURCE, in_channels=5, classes=5, resnet_checkpoint=RESNET_WEIGHT, vmamba_checkpoint=VMAMBA_WEIGHT)
smoke_model = smoke_model.cuda().train()
smoke_x = torch.randn(1, 5, 128, 128, device='cuda')
with torch.amp.autocast('cuda'):
    smoke_y = smoke_model(smoke_x)
assert smoke_y.shape == (1, 5, 128, 128) and torch.isfinite(smoke_y).all()
smoke_y.mean().backward()
assert loading['resnet18']['loaded_tensors'] > 100, loading
assert loading['vmamba_tiny']['loaded_tensors'] > 100, loading
assert loading['input_adaptation']['adapted_stems'] == 2
print('smoke test passed:', loading)
del smoke_model, smoke_x, smoke_y
torch.cuda.empty_cache()

In [ ]:
OUTPUT_DIR = Path('/kaggle/working')
command = [sys.executable, str(PROJECT_DIR / 'scripts/train_baseline.py'), '--model', config['model'], '--data-dir', str(DATA_DIR), '--output-dir', str(OUTPUT_DIR), '--rs3mamba-source-dir', str(RS3_SOURCE), '--pretrained-checkpoint', str(VMAMBA_WEIGHT), '--resnet-pretrained-checkpoint', str(RESNET_WEIGHT), '--seed', str(config['seed']), '--epochs', str(config['epochs']), '--batch-size', str(config['batch_size']), '--accum-steps', str(config['accum_steps']), '--num-workers', str(config['num_workers']), '--run-name', config['run_name'], '--skip-test-evaluation']
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR)

In [ ]:
RESULT_DIR = OUTPUT_DIR / f"result_{config['run_name']}"
metrics = json.loads((RESULT_DIR / 'metrics.json').read_text(encoding='utf-8'))
assert metrics['model'] == 'RS3Mamba' and metrics['encoder'] == 'resnet18_swsl+vmamba_tiny'
assert metrics['automatic_test_evaluation'] is False and metrics['test'] is None
assert metrics['pretrained_loading']['input_adaptation']['adapted_stems'] == 2
assert (RESULT_DIR / 'best_model.pth').is_file()
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('请下载完整目录:', RESULT_DIR)

In [ ]:
DIAGNOSTICS_DIR = RESULT_DIR / 'val_diagnostics'
eval_command = [sys.executable, str(PROJECT_DIR / 'scripts/evaluate_segmentation.py'), '--model', 'RS3Mamba', '--rs3mamba-source-dir', str(RS3_SOURCE), '--data-dir', str(DATA_DIR), '--checkpoint', str(RESULT_DIR / 'best_model.pth'), '--output-dir', str(DIAGNOSTICS_DIR), '--split', 'val', '--batch-size', '1', '--num-workers', str(config['num_workers'])]
print(' '.join(eval_command), flush=True)
subprocess.check_call(eval_command, cwd=PROJECT_DIR)
print('完整Val诊断已保存:', DIAGNOSTICS_DIR)